In [1]:
%env NODE_OPTIONS=--max-old-space-size=32768

env: NODE_OPTIONS=--max-old-space-size=32768


In [2]:
import torch
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import numpy as np
import neptune
# from accelerate import Accelerator
import evaluate
import os
# import ..utils
import sys
from datasets import Dataset
sys.path.append('/sise/home/urizlo/VuLLM_One_Stage')
from utils import Mistral_7B, Create_lora_mistral
from code_files.preprocess_data import Prepare_dataset_with_only_replace_mistral

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [3]:
checkpoint = "mistralai/Mistral-7B-v0.1"
model, tokenizer = Mistral_7B.create_model_and_tokenizer(checkpoint)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
%cd ..
%cd ..
%cd ..

/sise/home/urizlo/VuLLM_One_Stage/code_files/fine_tuning
/sise/home/urizlo/VuLLM_One_Stage/code_files
/sise/home/urizlo/VuLLM_One_Stage


In [5]:
path_trainset = "Datasets/vulgen_train_with_diff_lines_spaces.csv"
path_testset = "Datasets/vulgen_test_with_diff_lines_spaces.csv"
full_vulgen = True

train = Prepare_dataset_with_only_replace_mistral.get_train(path_trainset, full_vulgen)
test = Prepare_dataset_with_only_replace_mistral.get_test(path_testset, full_vulgen)
test_edits = Prepare_dataset_with_only_replace_mistral.get_edits(test)
train_edits = Prepare_dataset_with_only_replace_mistral.get_edits(train)
train['inputs'] = Prepare_dataset_with_only_replace_mistral.get_inputs(train)
train['outputs'] = Prepare_dataset_with_only_replace_mistral.get_outpus(train, train_edits)
test['inputs'] = Prepare_dataset_with_only_replace_mistral.get_inputs(test)
test['outputs'] = Prepare_dataset_with_only_replace_mistral.get_outpus(test, test_edits)
tokenized_train = Prepare_dataset_with_only_replace_mistral.tokenize(train, tokenizer)
tokenized_test = Prepare_dataset_with_only_replace_mistral.tokenize(test, tokenizer)
# tokenized_train = Dataset.from_dict(tokenized_train)
# tokenized_test = Dataset.from_dict(tokenized_test)

8135
978


In [6]:
def create_dataset_from_batch_encoding(batch_encoding, num_samples=100):
    # Convert BatchEncoding to dictionary
    data_dict = {key: batch_encoding[key][:num_samples] for key in batch_encoding.keys()}
    
    # Create a Dataset from the dictionary
    dataset = Dataset.from_dict(data_dict)
    return dataset


tokenized_test = create_dataset_from_batch_encoding(tokenized_test, 100)
tokenized_train = create_dataset_from_batch_encoding(tokenized_train, 100)



In [7]:
model = Create_lora_mistral.create_lora(model, rank=4, dropout=0.05)

trainable params: 10485760 || all params: 7252217856 || trainable%: 0.14458694165295632


In [8]:
# config evaluation metrics
metric = evaluate.load("sacrebleu")
google_bleu = evaluate.load("google_bleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    #ScareBleu
    results = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"sacreBleu": results["score"]}
    #GoogleBlue
    results = google_bleu.compute(predictions=decoded_preds, references=decoded_labels)
    result["googleBleu"] = results["google_bleu"]
    #Accuracy
    count = 0
    for p, l in zip(decoded_preds, decoded_labels):
        if p == l[0]:
            count += 1
    total_tokens = len(decoded_labels)
    accuracy = count / total_tokens
    result['accuracy'] = accuracy
    #Genaration length
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

In [14]:
# config env varibles
# NEPTUNE_API_TOKEN = os.environ.get("NEPTUNE_API_TOKEN")
# NEPTUNE_PROJECT = os.environ.get("NEPTUNE_PROJECT")
os.environ["NEPTUNE_API_TOKEN"] = 'eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiI4Y2VlNTFhZC1hODJkLTQ4NzItOTE0MS0yZmNkNWY3ZWE0MTEifQ=='
os.environ["NEPTUNE_PROJECT"] = 'zlotman/Localization-model'
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "true"

model.config.num_labels = tokenizer.vocab_size

In [15]:
# create trainer object
training_args = TrainingArguments(
    output_dir="saved_models/Mistral",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    adam_beta1=0.9,
    adam_beta2=0.95,
    adam_epsilon=1e-8,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    weight_decay=0.001,
    num_train_epochs=4,
    # predict_with_generate=True,
    bf16=True,
    tf32=True,
    remove_unused_columns=False,
    logging_dir="TensorBoard",
    do_train=True,
    do_eval=True,
    logging_strategy='epoch',
    # generation_max_length=810,
    # generation_num_beams=1,
    dataloader_num_workers=4,
    warmup_steps=57000,
    # report_to="no",
    lr_scheduler_type='linear',
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [16]:
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()

for i, batch in enumerate(tokenized_train):
    batch['input_ids'] = torch.tensor(batch['input_ids'], dtype=torch.long).unsqueeze(0).to(model.device)
    batch['attention_mask'] = torch.tensor(batch['attention_mask'], dtype=torch.bfloat16).unsqueeze(0).to(model.device)
    batch['labels'] = torch.tensor(batch['labels'], dtype=torch.long).unsqueeze(0).to(model.device)
        # Check labels for any out-of-bound values
    print("Label max:", batch['labels'].max())
    print("Label min:", batch['labels'].min())

    # Assuming 'num_classes' is the number of classes your model outputs
    num_classes = model.config.num_labels  # Adjust this according to your model's specific attribute or configuration
    print(num_classes)
    assert (batch['labels'] >= 0).all() and (batch['labels'] < num_classes).all(), "Labels are out of bound."
    with autocast():
        outputs = model(**batch)
        loss = outputs.loss
    scaler.scale(loss).backward()
    model.optimizer.step()
    scaler.update()
    model.optimizer.zero_grad()
    if i > 1:
        break  # just a quick test of a few batches

Label max: tensor(28734, device='cuda:0')
Label min: tensor(1, device='cuda:0')
32000


../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [0,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [1,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [2,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [3,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [4,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [5,0,0] Assertion `t >= 0 && t < n_classes` failed.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_f

RuntimeError: CUDA error: CUBLAS_STATUS_INTERNAL_ERROR when calling `cublasGemmEx( handle, opa, opb, m, n, k, &falpha, a, CUDA_R_16F, lda, b, CUDA_R_16F, ldb, &fbeta, c, CUDA_R_16F, ldc, CUDA_R_32F, CUBLAS_GEMM_DEFAULT_TENSOR_OP)`

In [11]:
trainer.train()

/home/urizlo/.conda/envs/vulinject/lib/python3.10/site-packages/neptune/common/warnings.py:62: NeptuneWarning: The following monitoring options are disabled by default in interactive sessions: 'capture_stdout', 'capture_stderr', 'capture_traceback', and 'capture_hardware_metrics'. To enable them, set each parameter to 'True' when initializing the run. The monitoring will continue until you call run.stop() or the kernel stops. Also note: Your source files can only be tracked if you pass the path(s) to the 'source_code' argument. For help, see the Neptune docs: https://docs.neptune.ai/logging/source_code/
  warnings.warn(


https://app.neptune.ai/zlotman/Localization-model/e/LOC-253


You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
../aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
